# Serve an LLM on your Mac's GPU as a Flyte app

An app with `gpu="AppleSilicon"` runs natively on this Mac and loads its model onto `mps`. The devbox still gives it a URL, logs and port forwarding. We serve Qwen3-1.7B behind an OpenAI-compatible API, measure it from a task, compare with MLX, and put a chat UI in front.

| Step | What it shows |
|---|---|
| 1 | Serve the model. Chat completions, streaming, tool calls. |
| 2 | Benchmark it from a CPU task, then the same prompts through MLX natively. |
| 3 | A chat UI that finds the model app by name. |

---

## Setup

This notebook runs **on your Mac**, next to an Apple Silicon devbox. It is not a Colab notebook: Colab has no Metal GPU and no devbox. (The code still runs there with `flyte run --local` and `ACCELERATOR=cpu`, just slowly.)

Before the first cell:

```bash
flyte start devbox --apple-silicon
cd tutorials/apple-silicon-serving
flyte create config --devbox
uv venv .venv --python 3.12 && source .venv/bin/activate && uv pip install -r requirements.txt pygments jupyter
```

In [ ]:
import os, platform
assert platform.machine() == 'arm64' and platform.system() == 'Darwin', 'This notebook expects an Apple Silicon Mac'
from utils.file_viewer import view_file
from utils.report_viewer import run_link, show_latest_local
!flyte get devbox --no-probes | head -5

### Running the steps

| | |
|---|---|
| `flyte run ...` | On the devbox. GPU tasks run natively on this Mac; every run gets a page in the UI at http://localhost:30080 with logs and reports. |
| `flyte run --local ...` | In this process, no devbox. On a Mac the GPU tasks still use `mps`. Reports land under `/tmp/flyte/metadata`; `show_latest_local()` renders the newest one. |

The cells below use the devbox. Each prints a run URL: open it for the live report.

### 1. Serve

The app loads the model, compiles the Metal kernels with one short generation, then takes traffic. The model downloads into your `~/.cache/huggingface` on first start.

In [ ]:
!python serve.py

In [ ]:
from openai import OpenAI
URL = "http://apple-silicon-llm-flytesnacks-development.localhost:30081"
client = OpenAI(base_url=f"{URL}/v1", api_key="not-used")
for chunk in client.chat.completions.create(model="Qwen/Qwen3-1.7B", stream=True, max_tokens=64,
        messages=[{"role": "user", "content": "Write a haiku about a laptop."}]):
    print(chunk.choices[0].delta.content or "", end="")

In [ ]:
# Tool calls: the model's <tool_call> blocks come back as OpenAI tool_calls.
resp = client.chat.completions.create(model="Qwen/Qwen3-1.7B", max_tokens=128,
    messages=[{"role": "user", "content": "What's the weather in Paris? Use the tool."}],
    tools=[{"type": "function", "function": {"name": "get_weather", "description": "Current weather for a city",
            "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}}}])
print(resp.choices[0].message.tool_calls)

In [ ]:
view_file("openai_server.py", title="the server (no Flyte in it)", show_path=True)
view_file("serve.py", title="the app environment", show_path=True)

### 2. Measure it from a task

`benchmark` is a CPU task that stays in its container and calls the app over the devbox network. `mlx_generate` asks for `gpu="AppleSilicon"` and runs mlx-lm on the Mac. `compare` runs both and draws the chart.

In [ ]:
!flyte run benchmark.py compare

In [ ]:
view_file("benchmark.py", title="calling an app from a task", show_path=True)
view_file("mlx_native.py", title="the Mac-native path", show_path=True)

### 3. A chat UI

A plain CPU app. It gets the model app's address as an `AppEndpoint` parameter.

In [ ]:
!python app_gradio.py

Open http://apple-silicon-llm-chat-flytesnacks-development.localhost:30081 in a browser.

### The same app on a cluster

```bash
ACCELERATOR=L40s python serve.py
```